## 搭建 RAG 系统
### 2.1 系统架构
#### 2.1.1 核心模块
索引：文档加载→文本分块→向量嵌入→向量存储。

检索：查询嵌入→相似度检索→提取上下文。

生成：提示工程→LLM 调度→输出后处理。

服务接口：REST API 端点→流式响应→监控指标。

#### 2.1.2 核心框架

|组件|技术选择|特点|
|---|---|---|
|嵌入模型|SentenceTransformers|句子级向量表示，可用于高效计算相似性。|
|向量数据库|Chroma|轻量级向量数据库，简单易用。<br>支持持久化存储与元数据管理。|
||FAISS|高效的相似度搜索库，支持多种搜索算法。<br>不具备持久化能力，需要手动持久化。|
||Milvus|支持大规模数据集的存储和检索，适合大规模<br>生产级应用。<br>提供完整的数据库功能，支持数据管理，持久<br>化存储。|
|LLM运行时|vLLM|高性能推理，连续批处理。|
|服务框架|FastAPI|异步支持，自动生成 OpenAPI 文档。|


#### 2.1.3 环境准备

In [1]:
# 直接使用系统 Python + pip（不需要 conda）
# 如果有多个 Python，建议先确认版本为 3.10+

In [2]:
#安装依赖：
# pip install langchain_community unstructured charset-normalizer==3.4.3 markdown pi_heif unstructured_inference pdf2image unstructured_pytesseract python-docx langchain_huggingface sentence-transformers langchain_chroma dashscope jieba langchain_openai faiss-cpu ragas bitsandbytes rank_bm25 pymysql sqlacodegen fastapi

### 2.2 索引过程
#### 2.2.1 文档加载
数据源可能来自多种格式的文件，如文本文档、Markdown，PDF等，首先需要对各种格式的文件进行处理，将其转化为可用的格式。
langchain_community.document_loaders中提供了多种格式的文档加载器，包括：
- TextLoader（文本文档加载）
- UnstructuredMarkdownLoader（Markdown加载）
- PyPDFLoader（PDF加载）
- UnstructuredPDFLoader（PDF加载）
- UnstructuredWordDocumentLoader（Word文档加载）
- WebBaseLoader（网站HTML加载）

1）加载文本文档

In [3]:
"""
TextLoader会将一个文档文件加载为一个Document对象，该对象有两个属性：
    metadata: 存储该文档的来源路径等元数据
    page_content: 存储文档的内容
"""

from langchain_community.document_loaders import TextLoader

text_documents = TextLoader("knowledge_base/sample.txt", encoding="utf-8").load()
print(text_documents)

d:\PY3\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'knowledge_base/sample.txt'}, page_content='始计第一\n\n孙子曰：兵者，国之大事，死生之地，存亡之道，不可不察也。\n\n故经之以五事，校之以计，而索其情：一曰道，二曰天，三曰地，四曰将，五曰法。\n\n道者，令民与上同意，可与之死，可与之生，而不畏危也；天者，阴阳、寒暑、时制也；地者，远近、险易、广狭、死生也；将者，智、信、仁、勇、严也；法者，曲制、官道、主用也。凡此五者，将莫不闻，知之者胜，不知者不胜。\n\n故校之以计，而索其情，曰：主孰有道？将孰有能？天地孰得？法令孰行？兵众孰强？士卒孰练？赏罚孰明？吾以此知胜负矣。\n\n将听吾计，用之必胜，留之；将不听吾计，用之必败，去之。\n\n计利以听，乃为之势，以佐其外。势者，因利而制权也。\n\n兵者，诡道也。故能而示之不能，用而示之不用，近而示之远，远而示之近。利而诱之，乱而取之，实而备之，强而避之，怒而挠之，卑而骄之，佚而劳之，亲而离之，攻其无备，出其不意。此兵家之胜，不可先传也。\n\n夫未战而庙算胜者，得算多也；未战而庙算不胜者，得算少也。多算胜，少算不胜，而况无算乎！吾以此观之，胜负见矣。\n\n作战第二\n\n孙子曰：凡用兵之法，驰车千驷，革车千乘，带甲十万，千里馈粮。则内外之费，宾客之用，胶漆之材，车甲之奉，日费千金，然后十万之师举矣。\n\n其用战也，贵胜，久则钝兵挫锐，攻城则力屈，久暴师则国用不足。夫钝兵挫锐，屈力殚货，则诸侯乘其弊而起，虽有智者，不能善其后矣。故兵闻拙速，未睹巧之久也。夫兵久而国利者，未之有也。故不尽知用兵之害者，则不能尽知用兵之利也。\n\n善用兵者，役不再籍，粮不三载，取用于国，因粮于敌，故军食可足也。国之贫于师者远输，远输则百姓贫；近于师者贵卖，贵卖则百姓竭，财竭则急于丘役。力屈财殚，中原内虚于家，百姓之费，十去其七；公家之费，破军罢马，甲胄矢弩，戟楯矛橹，丘牛大车，十去其六。\n\n故智将务食于敌，食敌一钟，当吾二十钟；萁秆一石，当吾二十石。故杀敌者，怒也；取敌之利者，货也。故车战，得车十乘以上，赏其先得者，而更其旌旗。车杂而乘之，卒善而养之，是谓胜敌而益强。\n\n故兵贵胜，不贵久。故知兵之将，民之司命。国家安危之主也。\n\n谋攻第三\n\n孙子曰：凡用兵之法，

2）加载Markdown

In [4]:
"""
UnstructuredMarkdownLoader可用于加载Markdown文件
    mode: 加载模式
        "single" 返回单个Document对象
        "elements" 按标题等元素切分文档
    strategy: 加载策略
        "fast" 快速粗粒度加载
        "hi_res" 细粒度加载，按标题层级、列表项、表格等结构细分
"""

from langchain_community.document_loaders import UnstructuredMarkdownLoader

md_documents = UnstructuredMarkdownLoader(
    "knowledge_base/sample.md",
    mode="elements",
    strategy="fast",
).load()
print(md_documents)

[Document(metadata={'source': 'knowledge_base/sample.md', 'emphasized_text_contents': ['Čeština', '∙', 'Deutsch', '∙', 'Ελληνικά', '∙', 'English', '∙', 'Español', '∙', 'Français', '∙', 'Indonesia', '∙', 'Italiano', '∙', '日本語', '∙', '한국어', '∙', 'polski', '∙', 'Português', '∙', 'Română', '∙', 'Русский', '∙', 'Slovenščina', '∙', 'Українська', '∙', '简体中文', '∙', '繁體中文'], 'emphasized_text_tags': ['i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i'], 'link_texts': ['Čeština', 'Deutsch', 'Ελληνικά', 'English', 'Español', 'Français', 'Indonesia', 'Italiano', '日本語', '한국어', 'polski', 'Português', 'Română', 'Русский', 'Slovenščina', 'Українська', '简体中文', '繁體中文'], 'link_urls': ['README-cs.md', 'README-de.md', 'README-el.md', 'README.md', 'README-es.md', 'README-fr.md', 'README-id.md', 'README-it.md', 'README-ja.md', 'README-ko.md', 'README-pl.md', 'README-pt.md', 'README-ro.md',

3）加载PDF

（1）使用 PyPDFLoader 解析文档

In [5]:
"""
PyPDFLoader
    支持页级拆分，每一页作为一个Document返回
    支持提取图像、提取布局
    extraction_mode: 提取模式
        "plain" 提取纯文本
        "layout" 提取布局
"""

from langchain_community.document_loaders import PyPDFLoader

pdf_documents = PyPDFLoader(
    "knowledge_base/sample.pdf",
    extraction_mode="layout",
).load()
print(pdf_documents)

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-07-24T17:46:07+08:00', 'title': '中国科学院国家天文台2023年度部门预算', 'author': 'MC SYSTEM', 'moddate': '2023-07-24T17:46:07+08:00', 'source': 'knowledge_base/sample.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1'}, page_content='中国科学院国家天文台\n          2023 年部门预算'), Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-07-24T17:46:07+08:00', 'title': '中国科学院国家天文台2023年度部门预算', 'author': 'MC SYSTEM', 'moddate': '2023-07-24T17:46:07+08:00', 'source': 'knowledge_base/sample.pdf', 'total_pages': 36, 'page': 1, 'page_label': '2'}, page_content='目             录\n一、中国科学院国家天文台基本情况 ..................................... 1\n    （一）单位职责 .................................................................... 1\n\n    （二）机构设置 .................................................................... 2\n二、中国科学院国家天文台                             2023    年

（2）使用 UnstructuredPDFLoader 解析文档

如果使用UnstructuredPDFLoader，需要先下载 Poppler 和 Tesseract OCR

Poppler 是一个开源的 PDF 文档处理库，用于渲染、解析和操作 PDF 文件。下载后将 .../poppler-24.08.0/Library/bin 添加到环境变量 Path 中即可

Tesseract OCR 提取图像中的文字，在安装时需要选择 Additional language data(download) 来添加中文包

安装后，将安装时设置的安装目录添加到环境变量 Path 中

使用 UnstructuredPDFLoader 解析文档：

In [6]:
"""
UnstructuredPDFLoader
    支持结构化提取，支持OCR
    仅当 PDF 文档中不存在文本时，才会应用 OCR
    mode: 加载模式
        "single" 返回单个Document对象
        "elements" 按标题等元素切分文档
    strategy: 加载策略
        "fast" 提取并处理文本
        "ocr_only" 先进行 OCR 处理，再处理原始文本
        "hi_res" 识别文档布局并处理，自动下载YOLOX模型（识别页面布局）
    infer_table_structure: 是否推断表格结构
        仅 hi_res 下起效
        如果为 True，提取出的表格元素会包含一个 text_as_html 元数据，将文本内容转换为 html 格式
    languages: OCR使用的语言
        需传入语言列表
        语言列表参考 https://github.com/tesseract-ocr/langdata
    更多参数详见 https://github.com/Unstructured-IO/unstructured/blob/main/unstructured/partition/pdf.py
"""

from langchain_community.document_loaders import UnstructuredPDFLoader

pdf_documents = UnstructuredPDFLoader(
    "knowledge_base/sample.pdf",
    mode="elements",
    strategy="hi_res",
    infer_table_structure=True,
    languages=["eng", "chi_sim"],
).load()
print(pdf_documents)

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


[Document(metadata={'source': 'knowledge_base/sample.pdf', 'detection_class_prob': 0.7602067589759827, 'coordinates': {'points': ((np.float64(349.54827880859375), np.float64(495.7198791503906)), (np.float64(349.54827880859375), np.float64(659.6992797851562)), (np.float64(568.7781372070312), np.float64(659.6992797851562)), (np.float64(568.7781372070312), np.float64(495.7198791503906))), 'system': 'PixelSpace', 'layout_width': 1654, 'layout_height': 2339}, 'last_modified': '2026-02-09T20:22:13', 'filetype': 'application/pdf', 'page_number': 1, 'file_directory': 'knowledge_base', 'filename': 'sample.pdf', 'category': 'Image', 'element_id': '298927504519a46201c07f73a7401597'}, page_content=''), Document(metadata={'source': 'knowledge_base/sample.pdf', 'coordinates': {'points': ((np.float64(339.0), np.float64(499.0)), (np.float64(339.0), np.float64(657.0)), (np.float64(1341.0), np.float64(657.0)), (np.float64(1341.0), np.float64(499.0))), 'system': 'PixelSpace', 'layout_width': 1654, 'layou

策略配置为hi_res时，会自动下载yolox模型用于页面布局的识别，会从huggingface下载到本地路径：C:\Users\<你的用户名>\.cache\huggingface\hub\

4）加载Word文档

In [7]:
"""
UnstructuredWordDocumentLoader
    适用于 .docx 和 .doc 文件
    mode: 加载模式
        "single" 返回单个Document对象
        "elements" 按标题等元素切分文档
    strategy: 加载策略
        "fast" 快速粗粒度加载
        "hi_res" 细粒度加载，按结构细分
"""

from langchain_community.document_loaders import UnstructuredWordDocumentLoader

word_documents = UnstructuredWordDocumentLoader(
    "knowledge_base/sample.docx",
    mode="elements",
    strategy="fast",
).load()
print(word_documents)

[Document(metadata={'source': 'knowledge_base/sample.docx', 'category_depth': 0, 'file_directory': 'knowledge_base', 'filename': 'sample.docx', 'last_modified': '2026-02-09T20:21:49', 'languages': ['zho'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'UncategorizedText', 'element_id': '02ab431f6036975771d14fbb197d8837'}, page_content='中华人民共和国民法典'), Document(metadata={'source': 'knowledge_base/sample.docx', 'category_depth': 0, 'file_directory': 'knowledge_base', 'filename': 'sample.docx', 'last_modified': '2026-02-09T20:21:49', 'languages': ['zho'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'UncategorizedText', 'element_id': '6bf8c1cd765f464d6755eba2fc1e1447'}, page_content='（2020年5月28日第十三届全国人民代表大会第三次'), Document(metadata={'source': 'knowledge_base/sample.docx', 'category_depth': 0, 'file_directory': 'knowledge_base', 'filename': 'sample.docx', 'last_modified': '2026-02-09T20:

5）通过网址链接加载HTML

In [8]:
"""
WebBaseLoader
    适用于网页
    web_paths: 网址序列
    bs_kwargs: 传给 BeautifulSoup 的解析参数
        parse_only 只提取指定标签的元素
"""
import bs4
from langchain_community.document_loaders import WebBaseLoader

web_documents = WebBaseLoader(
    web_paths=(
        "https://news.sina.com.cn/c/xl/2025-09-07/doc-infprmwn0510979.shtml",
    ),
    bs_kwargs={"parse_only": bs4.SoupStrainer(id="article")}, # 只提取正文区域
).load()
print(web_documents)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://news.sina.com.cn/c/xl/2025-09-07/doc-infprmwn0510979.shtml'}, page_content='\n\n\u3000\u30009月3日，纪念中国人民抗日战争暨世界反法西斯战争胜利80周年大会在天安门广场隆重举行。天安门城楼上，习近平总书记的重要讲话，6次提及这个光辉的字眼：和平。\n\u3000\u3000新华社9月5日播发重磅通讯《习近平引领中国和平发展的时代启示》。让我们通过这些故事，感悟习近平总书记发出的时代强音——“人类和平与发展的崇高事业必将胜利！”\n9月3日，纪念中国人民抗日战争暨世界反法西斯战争胜利80周年大会在北京隆重举行。中共中央总书记、国家主席、中央军委主席习近平发表重要讲话。新华社记者 燕雁 摄\n\u3000\u3000[和平如此珍贵，因为先烈的牺牲无价]\n\u3000\u3000“待风息波静，凯然而归，全家团聚……”\n\u3000\u3000李云鹏烈士留下的一封家信催人泪下。他是新四军“刘老庄连”82名烈士中唯一留下书信的人。战争无情、岁月易老，烈士当年对和平生活的向往，至今读来字字锥心。\n\u3000\u3000“正所谓‘诚既勇兮又以武，终刚强兮不可凌。身既死兮神以灵，魂魄毅兮为鬼雄。’”习近平总书记曾以《九歌·国殇》中的诗句赞扬“刘老庄连”等抗战英雄，表达崇高敬意。\n\u3000\u30002014年，我国通过立法确定中国人民抗日战争胜利纪念日，设立烈士纪念日、南京大屠杀死难者国家公祭日。每逢烈士纪念日，习近平总书记都向人民英雄敬献花篮，风雨无阻，行为世范。\n 2024年9月30日，习近平等党和国家领导人出席烈士纪念日向人民英雄敬献花篮仪式。新华社记者 殷博古 摄\n\u3000\u3000在中共中央政治局集体学习时，一字一句朗读赵一曼牺牲前写给儿子的绝笔信；感慨杨靖宇牺牲时胃里全是枯草、树皮、棉絮，“其事迹震撼人心”……\n\u3000\u3000一次次抚今追昔，一番番真情流露，凝聚成荡气回肠的坚定信念：“对一切为国家、为民族、为和平付出宝贵生命的人们，不管时代怎样变化，我们都要永远铭记他们的牺牲和奉献。”\n\u3000\u3000[和平如此珍贵，因为历史的教训惨痛]\n\u3000\u

6）封装函数，加载文件夹中多种文件类型

In [9]:
from langchain_community.document_loaders import (
    TextLoader,
    UnstructuredMarkdownLoader,
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader
)

def load_documents():
    """
    加载多种类型的文档，包括text、markdown、PDF和Word文档
    
    Returns:
        list: 包含所有加载文档的列表
    """
    # 加载文本文件
    text_documents = TextLoader(
        "knowledge_base/sample.txt",
        encoding="utf8"
    ).load()
    
    # 加载Markdown文件
    md_documents = UnstructuredMarkdownLoader(
        "knowledge_base/sample.md"
    ).load()
    
    # 加载PDF文件
    pdf_documents = UnstructuredPDFLoader(
        "knowledge_base/sample.pdf",
        mode="elements", # 元素模式
        strategy="hi_res", # 高分辨率策略
        # strategy="fast",
        languages=["eng", "chi_sim"], # 支持的语言：英文和简体中文
    ).load()
    
    # 加载Word文档
    word_documents = UnstructuredWordDocumentLoader(
        "knowledge_base/sample.docx"
    ).load()
    
    # 返回所有文档的列表
    return text_documents + md_documents + pdf_documents + word_documents

documents = load_documents()

7）文本清洗

对加载的文本进行清洗。去除 HTML 标签，去除多余的空格

后续使用的 Chroma 向量数据库，元数据只支持str、int、float、bool类型，因此需要将 metadata 中所有非 Chroma 支持类型的值转为 JSON 格式的字符串

In [10]:
import re
import json

def clean_content(documents: list):
    """文本清洗"""
    cleaned_docs = []
    
    for doc in documents:
        # 1、page_content处理：去除多余换行和空格
        text = doc.page_content
        
        # 将连续的换行符替换为两个换行符，正则表达式模式：r"\n{2,}"
        # # r"" 表示原始字符串（raw string），避免转义字符的特殊处理
        # # \n 表示换行符
        # # {2,} 是量词，表示前面的字符（换行符）出现 2 次或更多次
        text = re.sub(r"\n{2,}", "\n\n", text)
        text = text.strip()
        
        # 2、metadata处理：将所有非 Chroma 支持类型的值转为 JSON 格式字符串
        allowed_types = (str, int, float, bool)
        for key, value in doc.metadata.items():
            if not isinstance(value, allowed_types):
                try:
                    doc.metadata[key] = json.dumps(value, ensure_ascii=False, default=str)
                except (TypeError, ValueError):
                    # 如果 json.dumps 失败（如包含不可序列化对象），转为 str
                    doc.metadata[key] = str(value)
        
        # 3、更新文档内容
        doc.page_content = text
        cleaned_docs.append(doc)
    
    return cleaned_docs

#### 2.2.2 文本分块
加载后的文档往往过长，不适合直接作为 LLM 的上下文。因此在加载文档之后需要对文档内容进行分块，将一个长文档分割成多个块

如何对文档进行分块，往往决定了检索系统的下限。但是分块方式的选择与具体业务具有很强的关联性，针对不同业务不同文档往往需要定制不同的分块方式

langchain-text-splitters 中提供了多种文档分块方式，包括：
- CharacterTextSplitter（根据分隔符按字符拆分文档）
- RecursiveCharacterTextSplitter（递归使用多个分隔符按字符拆分文档）
- TokenTextSplitter（使用模型分词器将文本拆分为 token）
- SentenceTransformersTokenTextSplitter（使用句子模型分词器将文本拆分为 token）

文本分块时常用的两个参数 chunk_size 和 chunk_overlap，分别定义了块的大小和两个块之间重叠部分的大小。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

cleaned_docs = clean_content(documents)   # 这里调用清洗函数

# 文本分块
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "。"], # 分隔符列表
    chunk_size=500, # 每个块的最大长度
    chunk_overlap=50, # 每个块重叠的长度
)
texts = text_splitter.split_documents(cleaned_docs)

#### 2.2.3 向量嵌入与存储
使用 bge-base-zh-v1.5 模型将文本转换为向量
BGE 是一个向量模型，能够将任何文本映射为低维稠密向量，可用于检索、分类、聚类或语义搜索等任务

In [12]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# 加载嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="./bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    }, # 输出归一化向量，更适合余弦相似度计算
)

# 从 Document 中取出文本
page_content_list = [text.page_content for text in texts]
# 进行嵌入
embeddings = embedding_model.embed_documents(page_content_list)
# 打印嵌入结果
for i, (page_content, vector) in enumerate(zip(page_content_list, embeddings)):
    print("Text:\n", page_content)
    print("Embedding:\n", vector[:5])
    print()
    if i == 5:
        break

Text:
 始计第一

孙子曰：兵者，国之大事，死生之地，存亡之道，不可不察也。

故经之以五事，校之以计，而索其情：一曰道，二曰天，三曰地，四曰将，五曰法。

道者，令民与上同意，可与之死，可与之生，而不畏危也；天者，阴阳、寒暑、时制也；地者，远近、险易、广狭、死生也；将者，智、信、仁、勇、严也；法者，曲制、官道、主用也。凡此五者，将莫不闻，知之者胜，不知者不胜。

故校之以计，而索其情，曰：主孰有道？将孰有能？天地孰得？法令孰行？兵众孰强？士卒孰练？赏罚孰明？吾以此知胜负矣。

将听吾计，用之必胜，留之；将不听吾计，用之必败，去之。

计利以听，乃为之势，以佐其外。势者，因利而制权也。

兵者，诡道也。故能而示之不能，用而示之不用，近而示之远，远而示之近。利而诱之，乱而取之，实而备之，强而避之，怒而挠之，卑而骄之，佚而劳之，亲而离之，攻其无备，出其不意。此兵家之胜，不可先传也。
Embedding:
 [-0.020938647910952568, -0.022764025256037712, 0.0228037741035223, 0.015743987634778023, -0.010864011012017727]

Text:
 夫未战而庙算胜者，得算多也；未战而庙算不胜者，得算少也。多算胜，少算不胜，而况无算乎！吾以此观之，胜负见矣。

作战第二

孙子曰：凡用兵之法，驰车千驷，革车千乘，带甲十万，千里馈粮。则内外之费，宾客之用，胶漆之材，车甲之奉，日费千金，然后十万之师举矣。

其用战也，贵胜，久则钝兵挫锐，攻城则力屈，久暴师则国用不足。夫钝兵挫锐，屈力殚货，则诸侯乘其弊而起，虽有智者，不能善其后矣。故兵闻拙速，未睹巧之久也。夫兵久而国利者，未之有也。故不尽知用兵之害者，则不能尽知用兵之利也。

善用兵者，役不再籍，粮不三载，取用于国，因粮于敌，故军食可足也。国之贫于师者远输，远输则百姓贫；近于师者贵卖，贵卖则百姓竭，财竭则急于丘役。力屈财殚，中原内虚于家，百姓之费，十去其七；公家之费，破军罢马，甲胄矢弩，戟楯矛橹，丘牛大车，十去其六。
Embedding:
 [-0.0328071229159832, -0.013155095279216766, 0.00030313860042952, 0.009805133566260338, 0.00491

将文档嵌入并存入 Chroma 中

Chroma 中存储和查询的基本单位是 Collection，类似传统数据库中的表。每个 Collection 包含一组元素，每个元素包含以下内容：
- 唯一标示 ID
- 嵌入向量
- 嵌入向量对应文档
- 元数据键值对

In [13]:
from langchain_chroma import Chroma

# 嵌入并存储到向量数据库
vectorstore = Chroma.from_documents(
    texts, # 文档列表
    embedding_model, # 嵌入模型
    persist_directory="vectorstore", # 存储路径
)

通过 get 方法可以查看 Chroma 中的数据。

In [14]:
print(vectorstore.get().keys()) # 查看所有属性
print(vectorstore.get(include=["embeddings"])["embeddings"][:5, :5]) # 查看嵌入向量

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
[[-0.02093865 -0.02276403  0.02280377  0.01574399 -0.01086401]
 [-0.03280712 -0.0131551   0.00030314  0.00980513  0.00491961]
 [-0.01324996 -0.02942497 -0.01212257  0.0213747   0.00645994]
 [-0.01234541 -0.01730886 -0.00748602  0.0020273   0.01503307]
 [-0.02486498 -0.02161168  0.00156077  0.02090635 -0.02317911]]


### 2.3 检索过程
对用户的问题进行嵌入处理，转换成向量之后与向量数据库中的向量进行余弦相似度计算，返回最相似的若干个文档

首先加载嵌入模型和向量数据库

In [15]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 加载嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="./bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 初始化 Chroma 客户端
vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embedding_model,
)

检索向量数据库中与用户查询相似的文档

In [16]:
# 相似度检索
query = "不动产或者动产被占有人占有怎么办"
sim_docs = vectorstore.similarity_search(query, k=3) # 返回 3 条结果
for doc in sim_docs:
    print(doc)

page_content='第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。' metadata={'source': 'knowledge_base/sample.docx'}
page_content='第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。' metadata={'source': 'knowledge_base/sample.docx'}
page_content='第二百一十九条　利害关系人不得公开、非法使用权利人的不动产登记资料。

第二百二十条　权利人、利害关系人认为不动产登记簿记载的事项错误的，可以申请更正登记。不动产登记簿记载的权利人书面同意更正或者有证据证明登

也可以使用最大边际相关性检索（maximal marginal relevance，mmr），该检索方法的目的是在保证搜索结果相关性的同时，尽量减少结果之间的冗余和重复，提高多样性。它会首先返回一个最相关的文档，再迭代选择下一个文档，这个文档既要与查询相关，又不能和已经选出的文档太相似

In [17]:
# 最大边际相关性检索
query = "不动产或者动产被占有人占有怎么办"
sim_docs = vectorstore.max_marginal_relevance_search(query, k=3)
for doc in sim_docs:
    print(doc)

page_content='第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。' metadata={'source': 'knowledge_base/sample.docx'}
page_content='第二百一十九条　利害关系人不得公开、非法使用权利人的不动产登记资料。

第二百二十条　权利人、利害关系人认为不动产登记簿记载的事项错误的，可以申请更正登记。不动产登记簿记载的权利人书面同意更正或者有证据证明登记确有错误的，登记机构应当予以更正。

不动产登记簿记载的权利人不同意更正的，利害关系人可以申请异议登记。登记机构予以异议登记，申请人自异议登记之日起十五日内不提起诉讼的，异议登记失效。异议登记不当，造成权利人损害的，权利人可以向申请人请求损害赔偿。

第二百二十一条　当事人签订买卖房屋的协议或者签订其他不动产物权的协议，为保障将来实现物权，按照约定可以向登记机构申请预告登记。预告登记后，未经预告登记的权利人同意，处分该不动产的，不发生物权效力。

预告登记后，债权消灭或者自能够进行不动产登记之日起九十日内未申请登记的，预告登记失效。' metadata={'source': 'knowledge_base/sample.docx'}
page_content='第二百四十二条　法律规定专属于国家所有的不动产和动产，任何组织或者个人不能取得所有权。

第二百四十三条　为了公共利益的需要，依照法律规定的权限和程序可以征收集体所有的土地和组织、个人的房屋以及其他不动产。



也可以先获取向量存储检索器，再通过检索器进行检索

In [18]:
# 通过检索器检索
query = "不动产或者动产被占有人占有怎么办"
# 获取向量存储检索器
retriever = vectorstore.as_retriever(
    search_type="similarity", # 检索方式，similarity 或 mmr
    search_kwargs={"k": 3},
)
sim_docs = retriever.invoke(query)

for doc in sim_docs:
    print(doc)

page_content='第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。' metadata={'source': 'knowledge_base/sample.docx'}
page_content='第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。' metadata={'source': 'knowledge_base/sample.docx'}
page_content='第二百一十九条　利害关系人不得公开、非法使用权利人的不动产登记资料。

第二百二十条　权利人、利害关系人认为不动产登记簿记载的事项错误的，可以申请更正登记。不动产登记簿记载的权利人书面同意更正或者有证据证明登

## 2.4 生成过程
### 2.4.1 搭建检索生成链路
使用用户查询检索出相关文档之后，将文档和用户查询处理为 prompt 输入给 LLM。

可以使用 LCEL 将这些步骤组合成一个链条。LangChain Expression Language（LCEL，LangChain 表达式语言）是用于构建和组合链的一种简洁的声明式语法，允许像管道操作符一样将多个组件组合起来构成复杂的应用流程。并且支持诸如流式处理、并行处理和日志记录等开箱即用的功能

每个 LCEL 对象都实现了 Runnable 接口，该接口定义了一组公共的调用方法（invoke、batch、stream、ainvoke等）。这使得 LCEL 对象链也自动支持这些调用。也就是说，每个 LCEL 对象链本身也是一个 LCEL 对象。同时 Runnable 通过定义 __or__ 方法重载了 | 运算符，因此可以通过 | 将多个 Runnable 对象组合成一个 RunnableSequence

首先创建 .env 文件，在该文件中写入 LLM 的 API-Key

在开发环境中可以使用 .env，但在实际生产环境中建议使用实际的环境变量

TONGYI_API_KEY=我的 API-Key

首先加载嵌入模型与向量数据库，并创建检索器。

In [19]:
import os
import torch
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.tongyi import Tongyi
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.passthrough import RunnablePassthrough

# 加载嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="./bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 初始化 Chroma 客户端
vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embedding_model,
)

# 创建检索器
retriever = vectorstore.as_retriever(
    search_type="similarity", # 检索方式，similarity 或 mmr
    search_kwargs={"k": 3},
)

构建检索生成流程，使用检索器进行检索，LLM 再基于检索内容生成答案。

In [20]:
# 检索与生成链条
load_dotenv()
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")

# 将检索到的文档中的 page_content 取出组合到一起
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt 模板
prompt = PromptTemplate(
    input_variables=["context", "query"],
    template="""
你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：{context}

问题：{query}

回答：""",
)

# 大模型
llm = Tongyi(model="qwen-turbo", api_key=TONGYI_API_KEY)

# RAG 链条
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser() # 输出解析器，将输出解析为字符串
)

query = "不动产或者动产被占有人占有怎么办"
response = rag_chain.invoke(query)
print(response)


你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。

第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。

第二百一十九条　利害关系人不得公开、非法使用权利人的不动产登记资料。

第二百二十条　权利人、利害关系人认为不动产登记簿记载的事项错误的，可以申请更正登记。不动产登记簿记载的权利人书面同意更正或者有证据证明登记确有错误的，登记机构应当予以更正。

不动产登记簿记载的权利人不同意更正的，利害关系人可以申请异议登记。登记机构予以异议登记，申请人自

### 2.4.2 添加历史对话记录
在上述流程中，LLM 在交流过程中并不能记住历史的交流信息，这就导致模型不能在每次回复时关联历史交流信息，无法生成上下文连贯的回复

可以在每轮对话结束之后将本轮对话消息记录到历史记录中，并在下次对话时将历史记录作为 prompt 的一部分输入给 LLM

同时需要注意因为模型存在上下文长度限制，因此不能一股脑将所有历史记录都塞进 prompt 中，这样可能超出 LLM 的上下文长度限制，导致模型无法正常生成

In [ ]:
# 带历史对话记录
load_dotenv()
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")

# 将检索到的文档中的 page_content 取出组合到一起
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt 模板
prompt = PromptTemplate(
    input_variables=["context", "history", "query"],
    template="""
你是一个专业的中文问答助手，擅长基于提供的资料回答问题。
请仅根据以下背景资料以及历史消息回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：{context}

历史消息：[{history}]

问题：{query}

回答：""",
)

# 大模型
llm = Tongyi(model="qwen-turbo", api_key=TONGYI_API_KEY)

# 历史消息
history = []

# 格式化历史消息
def format_history(history):
    # 只保留最近 3 轮对话记录
    max_epoch = 3
    if len(history) > 2 * max_epoch:
        history = history[-2 * max_epoch :]
    return "\n".join([f"{i['role']}: {i['content']}" for i in history])

# RAG 链条
rag_chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["query"], k=3)),
        "history": lambda x: format_history(x["history"]),
        "query": lambda x: x["query"],
    }
    | prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser() # 输出解析器，将输出解析为字符串
)

query_list = ["不动产或者动产被人占有怎么办", "那要是被损毁了呢"]
for query in query_list:
    print(f"===== 查询: {query} =====")
    response = rag_chain.invoke({"query": query, "history": history})
    print(response, end="\n\n")
    history.extend(
        [
            {"role": "用户", "content": query},
            {"role": "助手", "content": response},
        ]
    )

===== 查询: 不动产或者动产被人占有怎么办 =====

你是一个专业的中文问答助手，擅长基于提供的资料回答问题。
请仅根据以下背景资料以及历史消息回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。

第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。

第二百一十九条　利害关系人不得公开、非法使用权利人的不动产登记资料。

第二百二十条　权利人、利害关系人认为不动产登记簿记载的事项错误的，可以申请更正登记。不动产登记簿记载的权利人书面同意更正或者有证据证明登记确有错误的，登记机构应当予以更正。

不动产登记簿记载的权利人不

### 2.4.3 重述用户消息
虽然模型已经能够利用上下文消息进行连贯对话，但是用户在根据上文继续提问时可能不会完整的表述所有的内容，而是使用代词指代或直接省略某些实体。如果使用这样的用户表述直接进行检索，可能无法准确检索到真正相关的文档，影响最终生成质量。

因此，在用户发送一个消息之后，我们可以根据历史对话记录重述用户消息，对用户消息进行指代消解或者补全缺失的实体，以保证后续检索的准确性。

在使用查询进行检索之前，先用 LLM 根据历史记录完善查询信息，再使用完善后的查询信息进行检索。如果不存在历史记录则直接使用查询信息进行检索。

In [29]:
# 重述用户消息
load_dotenv()
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")

# 将检索到的文档中的 page_content 取出组合到一起
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt 模板
prompt = PromptTemplate(
    input_variables=["context", "history", "query"],
    template="""
你是一个专业的中文问答助手，擅长基于提供的资料回答问题。
请仅根据以下背景资料以及历史消息回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：{context}

历史消息：[{history}]

问题：{query}

回答：""",
)

# 大模型
llm = Tongyi(model="qwen-turbo", api_key=TONGYI_API_KEY)

# 历史消息
history = []

# 格式化历史消息
def format_history(history):
    # 只保留最近 3 轮对话记录
    max_epoch = 3
    if len(history) > 2 * max_epoch:
        history = history[-2 * max_epoch :]
    return "\n".join([f"{i['role']}: {i['content']}" for i in history])

# rephrase Prompt 模板
rephrase_prompt = PromptTemplate(
    input_variables=["history", "query"],
    template="""
根据历史消息简要完善用户的问题，使其更加具体。只输出完善后的问题。

历史消息：[{history}]

问题：{query}
""",
)

# 重述链条：根据历史和当前 query 生成更具体问题
rephrase_chain = (
    {
        "history": lambda x: format_history(x["history"]),
        "query": lambda x: x["query"],
    }
    | rephrase_prompt
    | llm
    | StrOutputParser()
    | (lambda x: print(f"===== 重述后的查询: {x}=====") or x)
)

# Prompt 模板
prompt = PromptTemplate(
    input_variables=["context", "history", "query"],
    template="""
你是一个专业的中文问答助手，擅长基于提供的资料回答问题。
请仅根据以下背景资料以及历史消息回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：{context}

历史消息：[{history}]

问题：{query}

回答：""",
)

# RAG 链条
rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(
                rephrase_chain.invoke({"history": x["history"], "query": x["query"]})
                if x["history"]
                else x["query"],
                k=3,
            )
        ),
        "history": lambda x: format_history(x["history"]),
        "query": lambda x: x["query"],
    }
    | prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser() # 输出解析器，将输出解析为字符串
)

query_list = ["不动产或者动产被人占有怎么办", "那要是被损毁了呢"]
for query in query_list:
    print(f"===== 查询: {query} =====")
    response = rag_chain.invoke({"query": query, "history": history})
    print(response, end="\n\n")
    history.extend(
        [
            {"role": "用户", "content": query},
            {"role": "助手", "content": response},
        ]
    )

===== 查询: 不动产或者动产被人占有怎么办 =====

你是一个专业的中文问答助手，擅长基于提供的资料回答问题。
请仅根据以下背景资料以及历史消息回答问题，如无法找到答案，请直接回答“我不知道”。

背景资料：第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。

第四百五十七条　留置权人对留置财产丧失占有或者留置权人接受债务人另行提供担保的，留置权消灭。

第五分编　占有

第二十章　占有

第四百五十八条　基于合同关系等产生的占有，有关不动产或者动产的使用、收益、违约责任等，按照合同约定；合同没有约定或者约定不明确的，依照有关法律规定。

第四百五十九条　占有人因使用占有的不动产或者动产，致使该不动产或者动产受到损害的，恶意占有人应当承担赔偿责任。

第四百六十条　不动产或者动产被占有人占有的，权利人可以请求返还原物及其孳息；但是，应当支付善意占有人因维护该不动产或者动产支出的必要费用。

第四百六十一条　占有的不动产或者动产毁损、灭失，该不动产或者动产的权利人请求赔偿的，占有人应当将因毁损、灭失取得的保险金、赔偿金或者补偿金等返还给权利人；权利人的损害未得到足够弥补的，恶意占有人还应当赔偿损失。

第二百一十九条　利害关系人不得公开、非法使用权利人的不动产登记资料。

第二百二十条　权利人、利害关系人认为不动产登记簿记载的事项错误的，可以申请更正登记。不动产登记簿记载的权利人书面同意更正或者有证据证明登记确有错误的，登记机构应当予以更正。

不动产登记簿记载的权利人不